# Diabetes Disease Progression Prediction
**Domain:** Health | **Task:** Thiranex Task 4 - Real-world Data Project

Goal: predict disease progression one year after baseline for 442 diabetes patients using clinical measurements.

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
os.makedirs('output', exist_ok=True)

## 1. Load data
Features: age, sex, bmi, bp (blood pressure), s1-s6 (blood serum: total cholesterol, LDL, HDL, TCH ratio, log triglycerides, glucose). Target: disease progression score after 1 year.

In [ ]:
data = load_diabetes(as_frame=True)
df = data.frame.rename(columns={'target':'progression','s1':'total_chol','s2':'ldl','s3':'hdl','s4':'tch','s5':'ltg','s6':'glucose'})
print(df.shape); print(df.isna().sum().sum(), 'missing values')
df.describe().round(3)

## 2. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
sns.histplot(df['progression'], kde=True, ax=ax[0], color='teal'); ax[0].set_title('Distribution of Disease Progression')
sns.boxplot(data=df.drop(columns='progression'), ax=ax[1]); ax[1].set_title('Feature Spread (standardised)'); ax[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.savefig('output/1_distribution.png', dpi=150); plt.show()

In [ ]:
plt.figure(figsize=(9,7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap'); plt.tight_layout(); plt.savefig('output/2_correlation.png', dpi=150); plt.show()
corr = df.corr()['progression'].drop('progression').sort_values(ascending=False)
corr.round(3)

## 3. Model training & comparison
80/20 train-test split, 5-fold cross validation, 4 models.

In [ ]:
X, y = df.drop(columns='progression'), df['progression']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
models = {
 'Linear Regression': make_pipeline(StandardScaler(), LinearRegression()),
 'Ridge': make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
 'Random Forest': RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42),
 'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, learning_rate=0.03, max_depth=2, random_state=42),
}
rows, preds = [], {}
for name, m in models.items():
    m.fit(X_train, y_train); p = m.predict(X_test); preds[name] = p
    cv = cross_val_score(m, X, y, cv=5, scoring='r2')
    rows.append({'Model':name, 'R2':r2_score(y_test,p), 'RMSE':np.sqrt(mean_squared_error(y_test,p)),
                 'MAE':mean_absolute_error(y_test,p), 'CV_R2_mean':cv.mean(), 'CV_R2_std':cv.std()})
results = pd.DataFrame(rows).sort_values('R2', ascending=False).round(4).reset_index(drop=True)
results.to_csv('output/model_comparison.csv', index=False)
results

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14,4))
for a, metric in zip(ax, ['R2','RMSE','MAE']):
    sns.barplot(data=results, x=metric, y='Model', ax=a, palette='viridis'); a.set_title(metric)
plt.tight_layout(); plt.savefig('output/3_model_comparison.png', dpi=150); plt.show()

## 4. Best model analysis

In [ ]:
best_name = results.loc[0,'Model']; best = models[best_name]
print('Best model:', best_name)
if hasattr(best, 'feature_importances_'): imp = pd.Series(best.feature_importances_, index=X.columns)
else: imp = pd.Series(np.abs(best[-1].coef_), index=X.columns)
imp = (imp/imp.sum()).sort_values()
fig, ax = plt.subplots(1, 2, figsize=(13,4.5))
imp.plot.barh(ax=ax[0], color='darkorange'); ax[0].set_title(f'Feature Importance ({best_name})')
ax[1].scatter(y_test, preds[best_name], alpha=.7); lim=[y.min(), y.max()]; ax[1].plot(lim, lim, 'r--')
ax[1].set_xlabel('Actual'); ax[1].set_ylabel('Predicted'); ax[1].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig('output/4_best_model.png', dpi=150); plt.show()
imp.sort_values(ascending=False).round(4).to_csv('output/feature_importance.csv', header=['importance'])

## 5. Key Insights

In [ ]:
top = imp.sort_values(ascending=False).head(3)
print('KEY INSIGHTS'); print('-'*40)
print(f'1. Best model: {best_name} (R2={results.loc[0,"R2"]:.3f}, RMSE={results.loc[0,"RMSE"]:.1f}, MAE={results.loc[0,"MAE"]:.1f})')
print('2. Top 3 predictors:', ', '.join(f'{k} ({v:.0%})' for k,v in top.items()))
print(f'3. Strongest correlation with progression: {corr.idxmax()} ({corr.max():.2f})')
print(f'4. Average error is about {results.loc[0,"MAE"]:.0f} points on a score averaging {y.mean():.0f}')
print('5. Models explain under 60% of variance: clinical data alone cannot fully predict progression')